# Hotel Reservation System - Nigeria

This notebook implements a complete hotel reservation system with:

1. **OpenAI Integration**: Uses GPT-4o-mini with function calling for natural conversation
2. **Gradio Interface**: Clean chat UI with examples and a welcome message
3. **Pushover Notifications**: Sends booking confirmations to your phone
4. **Email Notifications**: Sends booking confirmations via email using SendGrid
5. **Nigerian Hotels Database**: Hotels in 7 major cities with real pricing in Naira
6. **3 Clarifying Questions**: The bot asks for state, date, and time before searching

### Tools Available:
- *search_hotels*: Find hotels by location and date
- *book_hotel*: Make a reservation
- *list_available_states*: Show all available locations
- *get_hotel_details*: Get details about a specific hotel
- *send_booking_summary*: Send push notification with booking details
- *send_email_summary*: Send booking confirmation via email

## Cell 1: Imports and Environment Setup

In [ ]:
import os
import json
import requests
from datetime import datetime, timedelta
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PUSHOVER_USER = os.getenv("PUSHOVER_USER")
PUSHOVER_TOKEN = os.getenv("PUSHOVER_TOKEN")
SENDGRID_API_KEY = os.getenv("SENDGRID_API_KEY")

SENDER_EMAIL = "olasogbayimika@gmail.com"
DEFAULT_RECIPIENT = "adebimpefolayemi1@gmail.com"

print("Environment loaded successfully!")
print(f"OpenAI Key present: {bool(OPENAI_API_KEY)}")
print(f"Pushover configured: {bool(PUSHOVER_USER and PUSHOVER_TOKEN)}")
print(f"SendGrid configured: {bool(SENDGRID_API_KEY)}")

## Cell 2: Nigerian Hotel Database

A simulated database of hotels across major Nigerian cities.

In [ ]:
NIGERIAN_HOTELS = {
    "Lagos": [
        {"name": "Eko Hotels & Suites", "rating": 5, "price_per_night": 85000, "rooms_available": 15, "amenities": ["Pool", "Spa", "Restaurant", "Gym", "WiFi"]},
        {"name": "Four Points by Sheraton Lagos", "rating": 4, "price_per_night": 65000, "rooms_available": 22, "amenities": ["Pool", "Restaurant", "Gym", "WiFi", "Business Center"]},
        {"name": "Radisson Blu Anchorage Hotel", "rating": 5, "price_per_night": 95000, "rooms_available": 10, "amenities": ["Pool", "Spa", "Restaurant", "Bar", "WiFi"]},
        {"name": "The George Hotel", "rating": 4, "price_per_night": 55000, "rooms_available": 8, "amenities": ["Restaurant", "Bar", "WiFi", "Parking"]},
        {"name": "Southern Sun Ikoyi", "rating": 4, "price_per_night": 48000, "rooms_available": 30, "amenities": ["Pool", "Restaurant", "Gym", "WiFi"]},
    ],
    "Abuja": [
        {"name": "Transcorp Hilton Abuja", "rating": 5, "price_per_night": 120000, "rooms_available": 25, "amenities": ["Pool", "Spa", "Restaurant", "Tennis", "WiFi"]},
        {"name": "Sheraton Abuja Hotel", "rating": 5, "price_per_night": 95000, "rooms_available": 18, "amenities": ["Pool", "Spa", "Restaurant", "Gym", "WiFi"]},
        {"name": "BON Hotel Abuja", "rating": 4, "price_per_night": 45000, "rooms_available": 35, "amenities": ["Restaurant", "Bar", "WiFi", "Parking"]},
        {"name": "The Envoy Hotel", "rating": 4, "price_per_night": 52000, "rooms_available": 12, "amenities": ["Pool", "Restaurant", "WiFi", "Gym"]},
        {"name": "Nordic Hotel Abuja", "rating": 3, "price_per_night": 28000, "rooms_available": 40, "amenities": ["Restaurant", "WiFi", "Parking"]},
    ],
    "Port Harcourt": [
        {"name": "Le Meridien Ogeyi Place", "rating": 5, "price_per_night": 75000, "rooms_available": 20, "amenities": ["Pool", "Spa", "Restaurant", "Gym", "WiFi"]},
        {"name": "Novotel Port Harcourt", "rating": 4, "price_per_night": 55000, "rooms_available": 28, "amenities": ["Pool", "Restaurant", "Bar", "WiFi"]},
        {"name": "Hotel Presidential", "rating": 4, "price_per_night": 42000, "rooms_available": 15, "amenities": ["Restaurant", "WiFi", "Parking", "Conference"]},
        {"name": "Swiss Spirit Hotel & Suites", "rating": 4, "price_per_night": 48000, "rooms_available": 22, "amenities": ["Pool", "Restaurant", "Gym", "WiFi"]},
    ],
    "Kano": [
        {"name": "Tahir Guest Palace", "rating": 4, "price_per_night": 35000, "rooms_available": 30, "amenities": ["Restaurant", "WiFi", "Parking", "Conference"]},
        {"name": "Bristol Palace Hotel", "rating": 3, "price_per_night": 22000, "rooms_available": 25, "amenities": ["Restaurant", "WiFi", "Parking"]},
        {"name": "Prince Hotel Kano", "rating": 3, "price_per_night": 18000, "rooms_available": 35, "amenities": ["Restaurant", "WiFi", "Parking"]},
    ],
    "Ibadan": [
        {"name": "Premier Hotel Ibadan", "rating": 4, "price_per_night": 38000, "rooms_available": 45, "amenities": ["Pool", "Restaurant", "WiFi", "Conference"]},
        {"name": "Kakanfo Inn and Conference Centre", "rating": 4, "price_per_night": 32000, "rooms_available": 28, "amenities": ["Pool", "Restaurant", "WiFi", "Conference"]},
        {"name": "De Ritz Hotel", "rating": 3, "price_per_night": 15000, "rooms_available": 20, "amenities": ["Restaurant", "WiFi", "Parking"]},
    ],
    "Enugu": [
        {"name": "Nike Lake Resort", "rating": 4, "price_per_night": 42000, "rooms_available": 35, "amenities": ["Pool", "Restaurant", "WiFi", "Lake View"]},
        {"name": "Best Western Plus Elomaz Hotel", "rating": 4, "price_per_night": 38000, "rooms_available": 22, "amenities": ["Pool", "Restaurant", "Gym", "WiFi"]},
        {"name": "Protea Hotel by Marriott", "rating": 4, "price_per_night": 45000, "rooms_available": 18, "amenities": ["Restaurant", "WiFi", "Gym", "Bar"]},
    ],
    "Calabar": [
        {"name": "Transcorp Hotels Calabar", "rating": 5, "price_per_night": 55000, "rooms_available": 30, "amenities": ["Pool", "Spa", "Restaurant", "WiFi"]},
        {"name": "Axari Hotel", "rating": 4, "price_per_night": 35000, "rooms_available": 25, "amenities": ["Pool", "Restaurant", "WiFi", "Parking"]},
        {"name": "Channel View Hotel", "rating": 3, "price_per_night": 18000, "rooms_available": 20, "amenities": ["Restaurant", "WiFi", "Parking"]},
    ],
}

AVAILABLE_STATES = list(NIGERIAN_HOTELS.keys())
print(f"Hotel database loaded with {len(AVAILABLE_STATES)} locations:")
for state in AVAILABLE_STATES:
    print(f"  - {state}: {len(NIGERIAN_HOTELS[state])} hotels")

## Cell 3: Notification Functions (Pushover & Email)

In [ ]:
def send_pushover_notification(message: str, title: str = "Hotel Booking - Nigeria") -> bool:
    """Send a push notification via Pushover API"""
    if not PUSHOVER_USER or not PUSHOVER_TOKEN:
        print("Pushover not configured, skipping notification")
        return False
    
    try:
        response = requests.post(
            "https://api.pushover.net/1/messages.json",
            data={
                "token": PUSHOVER_TOKEN,
                "user": PUSHOVER_USER,
                "message": message,
                "title": title,
            },
            timeout=10
        )
        if response.status_code == 200:
            print(f"Push notification sent: {message[:50]}...")
            return True
        else:
            print(f"Pushover error: {response.status_code}")
            return False
    except Exception as e:
        print(f"Pushover exception: {e}")
        return False


def send_email_notification(subject: str, html_body: str, recipient_email: str = None) -> dict:
    """Send an email with booking details using SendGrid"""
    if not SENDGRID_API_KEY:
        print("SendGrid not configured, skipping email")
        return {"success": False, "error": "Email service not configured"}
    
    try:
        sg = sendgrid.SendGridAPIClient(api_key=SENDGRID_API_KEY)
        from_email = Email(SENDER_EMAIL)
        to_email = To(recipient_email or DEFAULT_RECIPIENT)
        content = Content("text/html", html_body)
        mail = Mail(from_email, to_email, subject, content)
        
        response = sg.client.mail.send.post(request_body=mail.get())
        
        if response.status_code in [200, 201, 202]:
            print(f"Email sent successfully to {recipient_email or DEFAULT_RECIPIENT}")
            return {"success": True, "message": f"Email sent to {recipient_email or DEFAULT_RECIPIENT}"}
        else:
            print(f"SendGrid error: {response.status_code}")
            return {"success": False, "error": f"Email failed with status {response.status_code}"}
    except Exception as e:
        print(f"Email exception: {e}")
        return {"success": False, "error": str(e)}

print("Pushover and Email notification functions ready!")

## Cell 4: Hotel Tool Functions

These functions will be called by the OpenAI function calling mechanism.

In [ ]:
def search_hotels(state: str, check_in_date: str, check_out_date: str = None, 
                  max_price: int = None, min_rating: int = None) -> dict:
    """Search for available hotels in a Nigerian state"""
    state = state.title()
    
    if state not in NIGERIAN_HOTELS:
        return {
            "success": False,
            "error": f"'{state}' not found. Available: {', '.join(AVAILABLE_STATES)}"
        }
    
    try:
        check_in = datetime.strptime(check_in_date, "%Y-%m-%d")
        if check_out_date:
            check_out = datetime.strptime(check_out_date, "%Y-%m-%d")
        else:
            check_out = check_in + timedelta(days=1)
    except ValueError:
        return {"success": False, "error": "Invalid date format. Use YYYY-MM-DD."}
    
    num_nights = (check_out - check_in).days
    if num_nights <= 0:
        return {"success": False, "error": "Check-out must be after check-in."}
    
    hotels = NIGERIAN_HOTELS[state]
    results = []
    
    for hotel in hotels:
        if hotel["rooms_available"] <= 0:
            continue
        if max_price and hotel["price_per_night"] > max_price:
            continue
        if min_rating and hotel["rating"] < min_rating:
            continue
        
        total_price = hotel["price_per_night"] * num_nights
        results.append({
            "name": hotel["name"],
            "rating": hotel["rating"],
            "price_per_night": hotel["price_per_night"],
            "total_price": total_price,
            "rooms_available": hotel["rooms_available"],
            "amenities": hotel["amenities"]
        })
    
    return {
        "success": True,
        "state": state,
        "check_in": check_in_date,
        "check_out": check_out.strftime("%Y-%m-%d"),
        "num_nights": num_nights,
        "hotels": results,
        "count": len(results)
    }


def book_hotel(state: str, hotel_name: str, check_in_date: str, check_out_date: str,
               guest_name: str, num_rooms: int = 1, check_in_time: str = "14:00") -> dict:
    """Book a hotel room and send confirmation notification"""
    state = state.title()
    
    if state not in NIGERIAN_HOTELS:
        return {"success": False, "error": f"Invalid state: {state}"}
    
    hotel = None
    for h in NIGERIAN_HOTELS[state]:
        if h["name"].lower() == hotel_name.lower():
            hotel = h
            break
    
    if not hotel:
        return {"success": False, "error": f"Hotel '{hotel_name}' not found in {state}."}
    
    if hotel["rooms_available"] < num_rooms:
        return {"success": False, "error": f"Only {hotel['rooms_available']} rooms available."}
    
    try:
        check_in = datetime.strptime(check_in_date, "%Y-%m-%d")
        check_out = datetime.strptime(check_out_date, "%Y-%m-%d")
    except ValueError:
        return {"success": False, "error": "Invalid date format."}
    
    num_nights = (check_out - check_in).days
    total_price = hotel["price_per_night"] * num_nights * num_rooms
    
    import random
    booking_ref = f"NGH-{random.randint(100000, 999999)}"
    
    booking_details = {
        "success": True,
        "booking_reference": booking_ref,
        "guest_name": guest_name,
        "hotel_name": hotel["name"],
        "location": f"{state}, Nigeria",
        "check_in_date": check_in_date,
        "check_in_time": check_in_time,
        "check_out_date": check_out_date,
        "num_rooms": num_rooms,
        "num_nights": num_nights,
        "price_per_night": hotel["price_per_night"],
        "total_price": total_price,
        "amenities": hotel["amenities"]
    }
    
    notification_msg = f"""Booking Confirmed!
Ref: {booking_ref}
Guest: {guest_name}
Hotel: {hotel['name']}, {state}
Dates: {check_in_date} to {check_out_date}
Check-in Time: {check_in_time}
Rooms: {num_rooms}
Total: ₦{total_price:,}"""
    
    send_pushover_notification(notification_msg)
    
    return booking_details


def list_available_states() -> dict:
    """List all available Nigerian states/cities for hotel booking"""
    states_info = []
    for state in AVAILABLE_STATES:
        states_info.append({
            "state": state,
            "num_hotels": len(NIGERIAN_HOTELS[state])
        })
    return {"available_states": states_info}


def get_hotel_details(state: str, hotel_name: str) -> dict:
    """Get detailed information about a specific hotel"""
    state = state.title()
    
    if state not in NIGERIAN_HOTELS:
        return {"success": False, "error": f"Invalid state: {state}"}
    
    for hotel in NIGERIAN_HOTELS[state]:
        if hotel["name"].lower() == hotel_name.lower():
            return {
                "success": True,
                "name": hotel["name"],
                "location": f"{state}, Nigeria",
                "rating": hotel["rating"],
                "price_per_night": hotel["price_per_night"],
                "rooms_available": hotel["rooms_available"],
                "amenities": hotel["amenities"]
            }
    
    return {"success": False, "error": f"Hotel '{hotel_name}' not found in {state}."}

print("Hotel tool functions ready!")

## Cell 5: OpenAI Tool Definitions

JSON schemas for OpenAI function calling.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_hotels",
            "description": "Search for available hotels in a Nigerian state/city based on check-in date and preferences. Use this when the user wants to see available hotels.",
            "parameters": {
                "type": "object",
                "properties": {
                    "state": {
                        "type": "string",
                        "description": "The Nigerian state/city (e.g., Lagos, Abuja, Port Harcourt, Kano, Ibadan, Enugu, Calabar)"
                    },
                    "check_in_date": {
                        "type": "string",
                        "description": "Check-in date in YYYY-MM-DD format"
                    },
                    "check_out_date": {
                        "type": "string",
                        "description": "Check-out date in YYYY-MM-DD format (optional, defaults to next day)"
                    },
                    "max_price": {
                        "type": "integer",
                        "description": "Maximum price per night in Naira (optional)"
                    },
                    "min_rating": {
                        "type": "integer",
                        "description": "Minimum hotel rating 1-5 stars (optional)"
                    }
                },
                "required": ["state", "check_in_date"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "book_hotel",
            "description": "Book a hotel room for a guest. Use this when the user confirms they want to book a specific hotel.",
            "parameters": {
                "type": "object",
                "properties": {
                    "state": {
                        "type": "string",
                        "description": "The Nigerian state/city where the hotel is located"
                    },
                    "hotel_name": {
                        "type": "string",
                        "description": "The exact name of the hotel to book"
                    },
                    "check_in_date": {
                        "type": "string",
                        "description": "Check-in date in YYYY-MM-DD format"
                    },
                    "check_out_date": {
                        "type": "string",
                        "description": "Check-out date in YYYY-MM-DD format"
                    },
                    "guest_name": {
                        "type": "string",
                        "description": "Full name of the guest making the reservation"
                    },
                    "num_rooms": {
                        "type": "integer",
                        "description": "Number of rooms to book (default 1)"
                    },
                    "check_in_time": {
                        "type": "string",
                        "description": "Preferred check-in time in HH:MM format (default 14:00)"
                    }
                },
                "required": ["state", "hotel_name", "check_in_date", "check_out_date", "guest_name"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_available_states",
            "description": "List all Nigerian states/cities where hotels are available for booking",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_hotel_details",
            "description": "Get detailed information about a specific hotel including amenities and pricing",
            "parameters": {
                "type": "object",
                "properties": {
                    "state": {
                        "type": "string",
                        "description": "The Nigerian state/city where the hotel is located"
                    },
                    "hotel_name": {
                        "type": "string",
                        "description": "The name of the hotel"
                    }
                },
                "required": ["state", "hotel_name"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_booking_summary",
            "description": "Send a push notification with the booking summary to the user's phone via Pushover",
            "parameters": {
                "type": "object",
                "properties": {
                    "summary": {
                        "type": "string",
                        "description": "The booking summary message to send"
                    }
                },
                "required": ["summary"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_email_summary",
            "description": "Send an email with the booking summary. Use this when the user requests their booking details via email.",
            "parameters": {
                "type": "object",
                "properties": {
                    "subject": {
                        "type": "string",
                        "description": "The email subject line"
                    },
                    "booking_details": {
                        "type": "string",
                        "description": "The booking details to include in the email body (will be formatted as HTML)"
                    },
                    "recipient_email": {
                        "type": "string",
                        "description": "The recipient's email address (optional, uses default if not provided)"
                    }
                },
                "required": ["subject", "booking_details"],
                "additionalProperties": False
            }
        }
    }
]

def send_booking_summary(summary: str) -> dict:
    """Send a booking summary via Pushover notification"""
    success = send_pushover_notification(summary, title="Booking Summary")
    return {"success": success, "message": "Push notification sent" if success else "Failed to send push notification"}


def send_email_summary(subject: str, booking_details: str, recipient_email: str = None) -> dict:
    """Send a booking summary via email using SendGrid"""
    html_body = f"""
    <html>
    <body style="font-family: Arial, sans-serif; max-width: 600px; margin: 0 auto; padding: 20px;">
        <div style="background-color: #2e7d32; color: white; padding: 20px; text-align: center; border-radius: 10px 10px 0 0;">
            <h1 style="margin: 0;">Nigerian Hotels Reservation</h1>
            <p style="margin: 5px 0 0 0;">Your Booking Confirmation</p>
        </div>
        <div style="background-color: #f5f5f5; padding: 20px; border-radius: 0 0 10px 10px;">
            <pre style="white-space: pre-wrap; font-family: Arial, sans-serif; line-height: 1.6;">
{booking_details}
            </pre>
        </div>
        <div style="text-align: center; padding: 20px; color: #666;">
            <p>Thank you for choosing Nigerian Hotels!</p>
            <p style="font-size: 12px;">This is an automated email from Adebimpe, your AI booking assistant.</p>
        </div>
    </body>
    </html>
    """
    
    result = send_email_notification(subject, html_body, recipient_email)
    return result

print(f"Defined {len(tools)} tools for OpenAI function calling")

## Cell 6: Hotel Reservation Chatbot Class

The main chatbot class that handles conversations and tool calls.

In [ ]:
class HotelReservationBot:
    
    def __init__(self):
        self.openai = OpenAI()
        self.model = "gpt-4o-mini"
        self.available_tools = {
            "search_hotels": search_hotels,
            "book_hotel": book_hotel,
            "list_available_states": list_available_states,
            "get_hotel_details": get_hotel_details,
            "send_booking_summary": send_booking_summary,
            "send_email_summary": send_email_summary
        }
    
    def system_prompt(self):
        today = datetime.now().strftime("%Y-%m-%d")
        available_locations = ", ".join(AVAILABLE_STATES)
        
        return f"""You are a friendly and professional hotel reservation assistant for hotels in Nigeria. 
Your name is Olasogba, and you help guests find and book hotels across Nigeria.

Today's date is {today}.

IMPORTANT: Before searching for hotels, you MUST ask the user these 3 clarifying questions if not already provided:
1. **Which state/city in Nigeria?** (Available locations: {available_locations})
2. **What date do you want to check in?** (And check out date if staying multiple nights)
3. **What time do you prefer to check in?** (Standard check-in is 14:00/2PM)

Guidelines:
- Be warm, welcoming, and use Nigerian expressions appropriately (e.g., "Welcome o!", "No wahala")
- Always confirm details before making a booking
- Display prices in Naira (₦) with proper formatting
- When showing search results, format them nicely with ratings, prices, and amenities
- After a successful booking, use the send_booking_summary tool to send a push notification
- If the user requests the booking summary via EMAIL, use the send_email_summary tool instead
- Ask the user if they want the summary sent via push notification, email, or both
- If sending email, ask for their email address unless they've already provided it
- If the user asks about locations not in our database, politely list the available locations
- Help users compare hotels based on their budget and preferences

Remember: Always gather the 3 key pieces of information (state, date, time) before searching!"""
    
    def handle_tool_calls(self, tool_calls):
        """Process tool calls and return results"""
        results = []
        for tool_call in tool_calls:
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            
            print(f"Tool called: {tool_name}")
            print(f"Arguments: {arguments}")
            
            tool_func = self.available_tools.get(tool_name)
            if tool_func:
                result = tool_func(**arguments)
            else:
                result = {"error": f"Unknown tool: {tool_name}"}
            
            results.append({
                "role": "tool",
                "content": json.dumps(result),
                "tool_call_id": tool_call.id
            })
        
        return results
    
    def chat(self, message, history):
        """Main chat function for Gradio interface"""
        processed_history = [
            {"role": msg["role"], "content": msg["content"]} 
            for msg in history 
            if "role" in msg and "content" in msg
        ]
        
        messages = [
            {"role": "system", "content": self.system_prompt()}
        ] + processed_history + [
            {"role": "user", "content": message}
        ]
        
        done = False
        while not done:
            response = self.openai.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=tools
            )
            
            choice = response.choices[0]
            
            if choice.finish_reason == "tool_calls":
                assistant_message = choice.message
                tool_calls = assistant_message.tool_calls
                tool_results = self.handle_tool_calls(tool_calls)
                
                messages.append(assistant_message)
                messages.extend(tool_results)
            else:
                done = True
        
        return response.choices[0].message.content

bot = HotelReservationBot()
print("Hotel Reservation Bot initialized!")

## Cell 8: Launch Gradio Interface

The main Gradio chat interface for the hotel reservation system.

In [ ]:
welcome_message = """Welcome to Nigerian Hotels Reservation! I'm Olasogba, your hotel booking assistant.

I can help you find and book hotels across Nigeria. We have hotels in:
Lagos, Abuja, Port Harcourt, Kano, Ibadan, Enugu, and Calabar.

To get started, please tell me:
1. Which state/city are you traveling to?
2. What date would you like to check in?
3. What time do you prefer for check-in?

After booking, I can send your confirmation via push notification or email - just let me know your preference!

How can I help you today?"""

demo = gr.ChatInterface(
    fn=bot.chat,
    title="Nigerian Hotels Reservation System",
    description="Book hotels across Nigeria with Olasogba, your AI reservation assistant. Ask about available hotels, compare prices, and make reservations!",
    examples=[
        "I need a hotel in Lagos for next week",
        "What hotels are available in Abuja?",
        "Show me 5-star hotels in Port Harcourt",
        "I want to book a room for 3 nights in Calabar",
        "What locations do you have hotels in?"
    ],
    chatbot=gr.Chatbot(
        value=[{"role": "assistant", "content": welcome_message}],
        type="messages",
        height=500
    ),
    type="messages",
    theme=gr.themes.Soft(primary_hue="green"),
)

demo.launch()